In [0]:
from pyspark.sql import functions as F

# Ndan nje dataframe ne valid/invalid sipas nje kushti.
# Rreshtat invalid shkojne ne quarantine.<table> me arsyen.
# Kthen vetem rreshtat valid, per te vazhduar ne silver.
def split_and_quarantine(df, valid_condition, reject_reason, table_name, catalog, quarantine_schema):
    valid = df.filter(valid_condition)
    invalid = df.filter(~valid_condition).withColumn("reject_reason", F.lit(reject_reason))
    q_count = invalid.count()
    if q_count > 0:
        (invalid.write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(f"{catalog}.{quarantine_schema}.{table_name}"))
        print(f"  Quarantined {q_count:,} rows to {quarantine_schema}.{table_name} ({reject_reason})")
    return valid

In [0]:
# Shkruan nje dataframe ne silver si delta.
def write_to_silver(df, table_name, catalog, silver_schema, partition_col=None):
    writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(f"{catalog}.{silver_schema}.{table_name}")
    print(f"Wrote {df.count():,} rows to {silver_schema}.{table_name}")